# Notebook 03: GRU vs Transformer Training and Comparison

Both models are trained on SDAE-encoded features (16-dim latent).
Separate models trained per forecast horizon h ∈ {1, 3, 6, 12}.

Loss = full_profile_MSE + 1.0 × observed_mask_MSE

## Model architectures

**GRU** (21,192 params):
- Input: [B, 18, 16] → GRU(44, layers=2) → last hidden → MLP(22→6)

**Mini-Transformer** (36,838 params):
- Input: [B, 18, 16] → Linear(64) → SinPE → TransformerEncoder(1 layer, heads=4) → mean pool → MLP(32→6)

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, arch in enumerate(['gru', 'transformer']):
    for col, h in enumerate(cfg.HORIZONS):
        hist_path = cfg.OUT_METRICS / f'history_{arch}_h{h:02d}.npy'
        if not hist_path.exists(): continue
        hist = np.load(hist_path, allow_pickle=True).item()
        ax = axes[row, col]
        ax.plot(hist['train_loss'], color='steelblue', label='Train')
        ax.plot(hist['val_loss'],   color='coral',     label='Val')
        ax.set_title(f'{arch.upper()} | h={h}')
        ax.set_xlabel('Epoch')
        if col == 0: ax.set_ylabel('Loss')
        if row == 0 and col == 0: ax.legend(fontsize=8)
plt.suptitle('Training Curves: GRU vs Transformer across all horizons', fontsize=12)
plt.tight_layout(); plt.show()


## Results table

In [ ]:
results = pd.read_csv(cfg.OUT_METRICS / 'all_results.csv')
models = ['persistence', 'kinetic_prior', 'gru', 'transformer']
r = results[results['label'].isin(models)]
pivot = r.pivot_table(index='label', columns='horizon', values='obs_rmse').round(5)
print("Observed-Point RMSE by model and horizon:")
display(pivot)
pivot2 = r.pivot_table(index='label', columns='horizon', values='mape').round(2)
print("\nMAPE (%) by model and horizon:")
display(pivot2)


## Prediction visualizations

In [ ]:
from IPython.display import Image, display as disp
import ipywidgets as w
arch = 'transformer'; h = 1
img_path = cfg.OUT_FIGURES / f'05_pred_{arch}_h{h:02d}.png'
if img_path.exists(): disp(Image(str(img_path)))


In [ ]:
# Scatter plots h=1
disp(Image(str(cfg.OUT_FIGURES / '08_scatter_h01.png')))


## Horizon comparison

In [ ]:
disp(Image(str(cfg.OUT_FIGURES / '06_rmse_by_horizon.png')))


## Discussion: why does GRU underperform kinetic prior standalone?

The SDAE val MSE (4.0 vs train 0.15) reveals distribution shift: the validation
run (140313_1) operates in a different regime from the 6 training runs.
The SDAE latent representation is therefore less informative on val/test.

However, this is exactly why **fusion is critical**: even imperfect data-driven
predictions provide useful signal that kinetic prior alone cannot capture.
See Notebook 04 for fusion results.